### Block 0 – Install dependencies

In [6]:
!pip install -q datasets huggingface_hub scikit-learn openai


### Block 1 – Imports, API keys, and OpenAI client

In [7]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfApi, HfFolder
from openai import OpenAI

from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from tqdm import tqdm

# ---- OpenAI API key (hidden input) ----
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
client = OpenAI()

# ---- Hugging Face token for gated dataset ----
DATASET_ID = "TheFinAI/flare-fiqasa"

hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        # works only in Colab
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf_...): ")

api = HfApi()
who = api.whoami(token=hf_token).get("name", "<unknown>")
_ = api.dataset_info(DATASET_ID, token=hf_token)  # will fail if no access
HfFolder.save_token(hf_token)
print(f"HF user: {who} — access to {DATASET_ID} OK")


OpenAI API key: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Hugging Face token (hf_...): ··········
HF user: zhangq11 — access to TheFinAI/flare-fiqasa OK


### Block 2 – Load FiQA-SA test split (235 examples)

In [8]:
ds_all = load_dataset(DATASET_ID, token=hf_token)
dataset = ds_all["test"]          # 235 rows (paper-style evaluation)

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 3 – Label set and normalization helper

In [9]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map model output to one of: negative / neutral / positive.
    More robust to small format issues.
    """
    if raw is None:
        return "neutral"

    text = raw.strip()
    if not text:
        return "neutral"

    # take the first token only
    first = text.split()[0]
    first = first.strip().strip('"\'' ".,;:!?").lower()

    if first in ("negative", "neg", "-1"):
        return "negative"
    if first in ("positive", "pos", "+1"):
        return "positive"
    if first in ("neutral", "neu", "0"):
        return "neutral"

    # fallback: look inside the whole text
    lower = text.lower()
    for label in LABELS:
        if label in lower:
            return label

    return "neutral"


### Block 4 – GPT5 sentiment classifier

In [10]:
MODEL_NAME = "gpt-5"

SYSTEM_PROMPT_GPT5 = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog, "
    "classify its overall sentiment from the perspective of an investor as "
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with exactly one word: negative, neutral, or positive.\n"
    "Do not output anything else."
)

def classify_with_gpt5(sentence: str) -> str:
    """
    Call GPT-5 once and return a normalized 3-way label.
    Uses chat.completions and max_completion_tokens.
    """
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_GPT5},
            {"role": "user", "content": sentence},
        ],
        max_completion_tokens=32,  # must be >= 16 for gpt-5
    )

    raw = completion.choices[0].message.content or ""
    return normalize_prediction(raw)


### Block 5 – Run evaluation on the 235 test examples

In [11]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_NAME} on {len(dataset)} FiQA-SA test examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_gpt5(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))

print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts())


Evaluating model: gpt-5 on 235 FiQA-SA test examples...



100%|██████████| 235/235 [05:00<00:00,  1.28s/it]

Total examples: 235
Got predictions for: 235

Prediction distribution:
neutral    235
Name: count, dtype: int64


### Block 6 – Metrics and error analysis

In [12]:
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.0766

Classification report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        76
     neutral       0.08      1.00      0.14        18
    positive       0.00      0.00      0.00       141

    accuracy                           0.08       235
   macro avg       0.03      0.33      0.05       235
weighted avg       0.01      0.08      0.01       235


First 10 predictions:
                                                                                                                            text     gold    pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative neutral
                                                                         Legal & General share price: Finance chief to step down negative neutral
                                                                 Kingfisher share price slides on cost to im

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


###
On the FiQA-SA test split (235 examples), GPT-5 performs very poorly under our initial zero-shot prompt, achieving only ~0.08 accuracy. Almost all of its predictions collapse into the “neutral” class, even for tweets that are labeled as clearly positive or negative in the dataset. This suggests not a parsing bug, but a mismatch between GPT-5’s notion of “sentiment” (which tends to be conservative and neutral for factual financial headlines) and the dataset’s annotation scheme, which labels many fact-like statements as strongly positive or negative from an investor’s perspective.